In [2]:
%cd /kaggle/working
!git clone https://github.com/satwiksarkar/semeval_agent_risk_v1.git
%cd /kaggle/working/semeval_agent_risk_v1
!pip install -q -U transformers peft accelerate bitsandbytes

/kaggle/working
Cloning into 'semeval_agent_risk_v1'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 68 (delta 22), reused 57 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 53.49 KiB | 3.34 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/kaggle/working/semeval_agent_risk_v1


In [9]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [6]:
!find /kaggle/input -name "batch_*.pt"

/kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/dev/batch_00000.pt
/kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/train/batch_00000.pt


In [7]:
!python /kaggle/working/semeval_agent_risk_v1/experiments/phase_03_hybrid_gnn.py \
    --model_name_or_path "meta-llama/Meta-Llama-3-8B" \
    --train_data_dir "/kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/train" \
    --dev_data_dir "/kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/dev" \
    --output_dir "/kaggle/working/models/lora_adapters_dryrun" \
    --device_map "auto" \
    --load_in_4bit \
    --max_seq_len 1024 \
    --dry_run

12:51:15 [INFO] Target Compute Device: cuda:0 (Available GPUs: 2)
12:51:15 [INFO]   GPU [0]: Tesla T4 (VRAM: 14.6 GB)
12:51:15 [INFO]   GPU [1]: Tesla T4 (VRAM: 14.6 GB)
12:51:15 [INFO] Loading Tokenizer for meta-llama/Meta-Llama-3-8B...
12:51:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B "HTTP/1.1 200 OK"
12:51:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
12:51:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
12:51:16 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B "HTTP/1.1 200 OK"
12:51:16 [INFO] Loading batch file: batch_00000.pt
12:51:16 [INFO] Loaded 568 total trajectories from /kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/train
12:5

In [8]:
!python /kaggle/working/semeval_agent_risk_v1/experiments/phase_03_hybrid_gnn.py \
    --model_name_or_path "meta-llama/Meta-Llama-3-8B" \
    --train_data_dir "/kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/train" \
    --dev_data_dir "/kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/dev" \
    --output_dir "/kaggle/working/models/lora_adapters" \
    --device_map "auto" \
    --load_in_4bit \
    --epochs 5 \
    --micro_batch_size 2 \
    --grad_accum_steps 8 \
    --max_seq_len 1024 \
    --lr 2e-4

12:55:10 [INFO] Target Compute Device: cuda:0 (Available GPUs: 2)
12:55:10 [INFO]   GPU [0]: Tesla T4 (VRAM: 14.6 GB)
12:55:10 [INFO]   GPU [1]: Tesla T4 (VRAM: 14.6 GB)
12:55:10 [INFO] Loading Tokenizer for meta-llama/Meta-Llama-3-8B...
12:55:10 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B "HTTP/1.1 200 OK"
12:55:10 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
12:55:10 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
12:55:10 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B "HTTP/1.1 200 OK"
12:55:10 [INFO] Loading batch file: batch_00000.pt
12:55:11 [INFO] Loaded 568 total trajectories from /kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/train
12:5

In [9]:
!ls -lh /kaggle/working/models/lora_adapters/
!cat /kaggle/working/models/lora_adapters/training_manifest.json
!zip -r /kaggle/working/lora_adapters_trained.zip /kaggle/working/models/lora_adapters/

total 18M
-rw-r--r-- 1 root root 1.1K Nov 11 15:09 adapter_config.json
-rw------- 1 root root  14M Nov 11 15:09 adapter_model.safetensors
-rw-r--r-- 1 root root 4.3M Nov 11 15:09 fusion_heads.pt
-rw-r--r-- 1 root root 5.1K Nov 11 15:09 README.md
-rw-r--r-- 1 root root 1.1K Nov 11 15:43 training_manifest.json
{
  "task": "SemEval 2027 Task 10 (AgentRisk) - Phase 3 Training",
  "model_name_or_path": "meta-llama/Meta-Llama-3-8B",
  "lora_rank": 8,
  "lora_alpha": 16,
  "micro_batch_size": 2,
  "grad_accum_steps": 8,
  "effective_batch_size": 16,
  "device_map": "auto",
  "load_in_4bit": true,
  "history": [
    {
      "epoch": 1,
      "train_mae": 0.7231,
      "dev_mae": 0.4591,
      "is_best": true,
      "epoch_duration_seconds": 1997.43
    },
    {
      "epoch": 2,
      "train_mae": 0.4573,
      "dev_mae": 0.5063,
      "is_best": false,
      "epoch_duration_seconds": 2012.79
    },
    {
      "epoch": 3,
      "train_mae": 0.4108,
      "dev_mae": 0.4136,
      "is_best": tr

In [4]:
%cd /kaggle/working/semeval_agent_risk_v1
!git pull origin main
!pip install -q outlines

/kaggle/working/semeval_agent_risk_v1
From https://github.com/satwiksarkar/semeval_agent_risk_v1
 * branch            main       -> FETCH_HEAD
Already up to date.


In [6]:
%cd /kaggle/working/semeval_agent_risk_v1
!pip install -q -U transformers peft accelerate bitsandbytes outlines

/kaggle/working/semeval_agent_risk_v1


In [11]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [12]:
%%bash
ADAPTER_DIR=$(dirname "$(find -L /kaggle/input -name 'fusion_heads.pt' | head -n 1)")
DEV_DIR=$(dirname "$(find -L /kaggle/input -path '*/dev/batch_*.pt' | head -n 1)")
echo "ADAPTER_DIR: $ADAPTER_DIR"
echo "DEV_DIR:     $DEV_DIR"

python /kaggle/working/semeval_agent_risk_v1/experiments/phase_04_inference.py \
    --mode eval \
    --adapter_dir "$ADAPTER_DIR" \
    --data_path "$DEV_DIR" \
    --output_dir "/kaggle/working/outputs" \
    --device_map "auto" \
    --load_in_4bit \
    --max_seq_len 1024

cat /kaggle/working/outputs/phase4_dev_eval_report.json

ADAPTER_DIR: /kaggle/input/datasets/satwiksarkar06/agent-risk-lora-adapters/models/lora_adapters
DEV_DIR:     /kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/dev
{
  "num_samples": 143,
  "continuous_mae": 0.3966,
  "integer_mae": 0.3427,
  "exact_match_accuracy_pct": 68.53,
  "within_1_accuracy_pct": 97.2,
  "per_class_metrics": {
    "1": {
      "count": 66,
      "int_mae": 0.0303,
      "exact_acc_pct": 96.97
    },
    "2": {
      "count": 8,
      "int_mae": 0.75,
      "exact_acc_pct": 37.5
    },
    "3": {
      "count": 32,
      "int_mae": 0.9062,
      "exact_acc_pct": 18.75
    },
    "4": {
      "count": 35,
      "int_mae": 0.3429,
      "exact_acc_pct": 65.71
    },
    "5": {
      "count": 2,
      "int_mae": 0.0,
      "exact_acc_pct": 100.0
    }
  },
  "confusion_matrix_1_to_5": [
    [
      64,
      2,
      0,
      0,
      0
    ],
    [
      4,
      3,
      0,
      1,
      0
    ],
    [
      0,
      1,
     

17:03:12 [INFO] Loading batch file: batch_00000.pt
17:03:12 [INFO] Loaded 143 total trajectories from /kaggle/input/datasets/satwiksarkar06/agent-risk-phase2/data/processed/phase_2_verified/dev
17:03:12 [INFO] Loading Tokenizer for meta-llama/Meta-Llama-3-8B...
17:03:12 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B "HTTP/1.1 200 OK"
17:03:12 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
17:03:12 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
17:03:12 [INFO] HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B "HTTP/1.1 200 OK"
17:03:12 [INFO] Loading AutoModelForCausalLM (meta-llama/Meta-Llama-3-8B) in shared-VRAM mode (4bit=True, device_map=auto)...
17:03:12 [INFO] HTTP Request: GET https://huggingf

In [13]:
!zip -r /kaggle/working/phase4_outputs.zip /kaggle/working/outputs/

  adding: kaggle/working/outputs/ (stored 0%)
  adding: kaggle/working/outputs/phase4_dev_eval_report.json (deflated 69%)
  adding: kaggle/working/outputs/phase4_dev_submission.csv (deflated 82%)
  adding: kaggle/working/outputs/phase4_dev_predictions.jsonl (deflated 95%)
